# Notebook 8 – Data Manipulation

Data manipulation is the process of modifying, transforming, and organizing data to make it clean, consistent, and useful for analysis. In Pandas, we can rename columns, add or remove columns, update values, apply functions, replace unwanted values, and handle duplicate records. These operations are commonly used in real-world data analysis, business reporting, data cleaning, and AI/ML data preprocessing.
### The Dataset We'll Use Throughout

- **Real-world / Business angle:** The company's ops/marketing team needs to clean, relabel, and enrich this customer export before it goes into dashboards and campaigns.
- **AI/ML angle:** The exact same cleaning steps — renaming messy columns, dropping bad rows, engineering new columns, encoding categories — are the standard **data preprocessing / feature engineering** steps before training a churn model.

In [1]:
import pandas as pd
import numpy as np
data = {
    "CustID":        [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 108],
    "Cust Name":     ["Aarav", "Priya", "Rahul", "Sneha", "Vikram",
                       "Ananya", "Karthik", "Divya", "Manoj", "Lakshmi", "Divya"],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South", "West"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze", "Gold"],
    "total_orders":  [42, 15, 30, 5, 22, 60, 8, 35, 18, 3, 35],
    "total_spend":   [125000, 32000, 98000, 8000, 45000,
                       210000, 12000, 87000, 39000, 4500, 87000],
    "last_order_value": [3200, 1500, 4200, 900, 2100,
                          5000, 1100, 3900, 1800, 700, 3900],
    "churn_risk":    ["Low", "Medium", "Low", "High", "Medium",
                        "Low", "High", "Low", "Medium", "High", "Low"]
}
df = pd.DataFrame(data)
df

,CustID,Cust Name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


**Note:** Notice the columns have inconsistent naming (`CustID`, `Cust Name` vs lowercase `region`), and row index `10` is an exact duplicate of customer 108 (*Divya*) — both are common real-world data-quality issues we'll clean up below.

## 1. Rename Columns

### Concept Explanation
`rename(columns={...})` relabels one or more column names without touching the data itself. Use `inplace=True` to modify the original DataFrame, or capture the return value (the default, safer approach).

### Business + AI/ML Example
The raw export has inconsistent column names (`CustID`, `Cust Name`). Before this goes into a dashboard — or into a churn model's feature pipeline — the team standardizes all column names to a clean, consistent `snake_case` format, since most BI tools and ML libraries expect consistent naming.

In [2]:
df = df.rename(columns={"CustID": "customer_id", "Cust Name": "customer_name"})
df.columns.tolist()

['customer_id',
 'customer_name',
 'region',
 'membership',
 'total_orders',
 'total_spend',
 'last_order_value',
 'churn_risk']

**Output Explanation:** The column list now shows `customer_id` and `customer_name` instead of the old `CustID` / `Cust Name`. All other columns were left unchanged since we only specified the two we wanted to rename — this is the safest way to rename, as it avoids accidentally mismatching a full column list.

## 2. Drop Rows

### Concept Explanation
`drop(index_labels, axis=0)` (or the shortcut `df.drop([...])`) removes specific rows by their index label. This is different from *filtering* (Notebook 7) — here we're explicitly removing known rows rather than keeping rows that match a condition.

### Business + AI/ML Example
The business has flagged **customer 104 (Sneha)** as a test/dummy account that was accidentally included in the export and must be removed before reporting. Removing known bad rows like this is also a standard first step before training an ML model — bad rows would otherwise corrupt the churn model's learned patterns.

In [2]:
df = df.drop(index=3)  
df

,CustID,Cust Name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High
10,108,Divya,West,Gold,35,87000,3900,Low


**Output Explanation:** Row label `3` (Sneha) is gone, and the DataFrame now has one fewer row (10 remaining, including the duplicate we'll handle later). The remaining index labels (0,1,2,4,5,...) are **not** renumbered automatically — `drop()` just removes the row, it doesn't reset the index.

## 3. Drop Columns

### Concept Explanation
`drop(columns=[...])` (or `axis=1`) removes one or more entire columns. Like row-dropping, it returns a new DataFrame by default unless `inplace=True` is used.

### Business + AI/ML Example
The `last_order_value` column was only needed for a one-off report and isn't relevant to the ongoing dashboard, so the business drops it. In ML, dropping columns is exactly how you'd remove a **leaky or irrelevant feature** before training — e.g., a feature that wouldn't be available at prediction time.

In [4]:
df_reduced = df.drop(columns=["last_order_value"])
df_reduced.columns.tolist()

['customer_id',
 'customer_name',
 'region',
 'membership',
 'total_orders',
 'total_spend',
 'churn_risk']

**Output Explanation:** `last_order_value` no longer appears in the column list. We assigned the result to a new variable `df_reduced` rather than overwriting `df`, since we still want the full dataset later — a common practice when you're not sure yet whether you'll need the dropped column.

## 4. Insert Columns

### Concept Explanation
New columns can be added simply via `df["new_col"] = ...`, or inserted at a **specific position** using `df.insert(loc, column, value)`. `insert()` is useful when column order matters (e.g., for readability or a downstream tool that expects a specific layout).

### Business + AI/ML Example
The business wants an `avg_order_value` metric (spend ÷ orders) placed right after `total_spend` for easy reading on a dashboard. This is also a textbook example of **feature engineering** in ML — deriving a new, often more predictive feature (average order value) from existing raw columns.

In [5]:
avg_order_value = (df["total_spend"] / df["total_orders"]).round(2)

# Insert at position 6 (right after 'total_spend'), so it reads naturally left-to-right
position = df.columns.get_loc("total_spend") + 1
df.insert(position, "avg_order_value", avg_order_value)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,avg_order_value,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,2976.19,3200,Low
1,102,Priya,North,Silver,15,32000,2133.33,1500,Medium
2,103,Rahul,South,Gold,30,98000,3266.67,4200,Low
4,105,Vikram,East,Silver,22,45000,2045.45,2100,Medium
5,106,Ananya,South,Gold,60,210000,3500.00,5000,Low
6,107,Karthik,North,Bronze,8,12000,1500.00,1100,High
7,108,Divya,West,Gold,35,87000,2485.71,3900,Low
8,109,Manoj,East,Silver,18,39000,2166.67,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,1500.00,700,High
10,108,Divya,West,Gold,35,87000,2485.71,3900,Low


**Output Explanation:** A new `avg_order_value` column now sits immediately after `total_spend`, showing each customer's average spend per order (e.g., Aarav: 125000/42 ≈ 2976.19). Placing it at a specific position (rather than always at the end) keeps related columns visually grouped, which matters for both dashboards and human-readable feature tables.

## 5. `assign()`

### Concept Explanation
`assign(new_col=expression)` returns a **new DataFrame** with additional (or modified) columns, without mutating the original — useful for chaining multiple transformations fluently in one statement.

### Business + AI/ML Example
The business wants to flag customers as **"VIP"** if their total spend exceeds 80,000, without permanently altering the working DataFrame yet (e.g., while still experimenting with the threshold). This is also how a data scientist commonly derives a **binary label/feature** on the fly during exploratory analysis for a churn model.

In [6]:
df_with_vip = df.assign(
    is_vip=lambda x: x["total_spend"] > 80000,
    spend_per_order_rank=lambda x: x["avg_order_value"].rank(ascending=False).astype(int)
)
df_with_vip[["customer_name", "total_spend", "is_vip", "spend_per_order_rank"]]

,customer_name,total_spend,is_vip,spend_per_order_rank
0,Aarav,125000,True,3
1,Priya,32000,False,7
2,Rahul,98000,True,2
4,Vikram,45000,False,8
5,Ananya,210000,True,1
6,Karthik,12000,False,9
7,Divya,87000,True,4
8,Manoj,39000,False,6
9,Lakshmi,4500,False,9
10,Divya,87000,True,4


**Output Explanation:** Two new columns were added — `is_vip` (True/False based on the 80,000 threshold) and `spend_per_order_rank` (rank position by average order value, 1 = highest). Because `assign()` returns a new object, the original `df` is untouched; we can freely discard `df_with_vip` if the VIP threshold turns out to need adjusting.

## 6. `apply()`

### Concept Explanation
`apply(func)` runs a function along an axis of the DataFrame — on each **column** (default, `axis=0`), each **row** (`axis=1`), or each value of a **Series**. It's the most flexible transformation tool, used when a simple vectorized operation isn't enough (e.g., logic spanning multiple columns).

### Business + AI/ML Example
The business wants a single **customer tier label** that combines membership level *and* order count using custom business logic (something no single built-in function does). Writing this kind of row-wise custom rule with `apply()` is exactly how you'd hand-craft a derived categorical feature for a churn model too.

In [4]:
df = df.rename(columns={"CustID": "customer_id", "Cust Name": "customer_name"})
df.columns.tolist()

['customer_id',
 'customer_name',
 'region',
 'membership',
 'total_orders',
 'total_spend',
 'last_order_value',
 'churn_risk',
 'customer_tier']

In [5]:
def tier_label(row):
    if row["membership"] == "Gold" and row["total_orders"] > 30:
        return "Platinum Candidate"
    elif row["membership"] == "Gold":
        return "Gold"
    elif row["total_orders"] > 20:
        return "Rising Silver/Bronze"
    else:
        return row["membership"]

df["customer_tier"] = df.apply(tier_label, axis=1)
df[["customer_name", "membership", "total_orders", "customer_tier"]]

,customer_name,membership,total_orders,customer_tier
0,Aarav,Gold,42,Platinum Candidate
1,Priya,Silver,15,Silver
2,Rahul,Gold,30,Gold
4,Vikram,Silver,22,Rising Silver/Bronze
5,Ananya,Gold,60,Platinum Candidate
6,Karthik,Bronze,8,Bronze
7,Divya,Gold,35,Platinum Candidate
8,Manoj,Silver,18,Silver
9,Lakshmi,Bronze,3,Bronze
10,Divya,Gold,35,Platinum Candidate


In [6]:
if "customer_name" not in df.columns and "Cust Name" in df.columns:
    df = df.rename(columns={"CustID": "customer_id", "Cust Name": "customer_name"})

def tier_label(row):
    if row["membership"] == "Gold" and row["total_orders"] > 30:
        return "Platinum Candidate"
    elif row["membership"] == "Gold":
        return "Gold"
    elif row["total_orders"] > 20:
        return "Rising Silver/Bronze"
    else:
        return row["membership"]

df["customer_tier"] = df.apply(tier_label, axis=1)
df[["customer_name", "membership", "total_orders", "customer_tier"]]

,customer_name,membership,total_orders,customer_tier
0,Aarav,Gold,42,Platinum Candidate
1,Priya,Silver,15,Silver
2,Rahul,Gold,30,Gold
4,Vikram,Silver,22,Rising Silver/Bronze
5,Ananya,Gold,60,Platinum Candidate
6,Karthik,Bronze,8,Bronze
7,Divya,Gold,35,Platinum Candidate
8,Manoj,Silver,18,Silver
9,Lakshmi,Bronze,3,Bronze
10,Divya,Gold,35,Platinum Candidate


**Output Explanation:** `
First, it makes sure the columns are renamed properly (fixes the earlier error).
                                                       
tier_label() looks at each row's membership and total_orders to decide a tier.

apply(..., axis=1) runs that function on every row and saves the result as customer_tier.

Last line just shows the result.

## 7. `map()`

### Concept Explanation
`map()` (called on a **Series**, not a whole DataFrame) transforms each value using a dictionary, a Series, or a function — one value in, one value out. It's simpler and faster than `apply()` for **single-column, value-by-value** substitutions.

### Business + AI/ML Example
The business wants `churn_risk` (currently text: Low/Medium/High) converted into a numeric **risk score** for a dashboard color scale. Encoding a categorical column into numbers this way is also a **required preprocessing step** before feeding data into most ML algorithms, which need numeric input.

In [7]:
risk_score_map = {"Low": 1, "Medium": 2, "High": 3}
df["churn_risk_score"] = df["churn_risk"].map(risk_score_map)
df[["customer_name", "churn_risk", "churn_risk_score"]]

,customer_name,churn_risk,churn_risk_score
0,Aarav,Low,1
1,Priya,Medium,2
2,Rahul,Low,1
4,Vikram,Medium,2
5,Ananya,Low,1
6,Karthik,High,3
7,Divya,Low,1
8,Manoj,Medium,2
9,Lakshmi,High,3
10,Divya,Low,1


**Output Explanation:** Every text label in `churn_risk` was replaced by its corresponding number via the dictionary lookup — `"Low"` → 1, `"Medium"` → 2, `"High"` → 3. This numeric `churn_risk_score` column could now be used directly as a target/feature in a machine learning model, whereas the text version could not.

## 8. `replace()`

### Concept Explanation
`replace(to_replace, value)` substitutes specific **values** (not just via a full dict-map on one column) anywhere in a Series or DataFrame — useful for fixing typos, standardizing inconsistent category labels, or handling placeholder values like `-1` or `"unknown"`.

### Business + AI/ML Example
The `region` column has an inconsistent entry style, and the business wants `"North"` and `"East"` regions relabeled to a combined `"North-East Zone"` for a new regional sales structure. Standardizing inconsistent category values like this is also crucial before ML training, since a model would otherwise treat inconsistent labels as entirely different categories.

In [8]:
df["region"] = df["region"].replace({"North": "North-East Zone", "East": "North-East Zone"})
df[["customer_name", "region"]]

,customer_name,region
0,Aarav,South
1,Priya,North-East Zone
2,Rahul,South
4,Vikram,North-East Zone
5,Ananya,South
6,Karthik,North-East Zone
7,Divya,West
8,Manoj,North-East Zone
9,Lakshmi,South
10,Divya,West


**Output Explanation:** Every occurrence of `"North"` or `"East"` in the `region` column was replaced with `"North-East Zone"`, while `"South"` and `"West"` were left unchanged. This kind of value consolidation reduces the number of distinct categories a downstream dashboard filter (or an ML model's one-hot encoding) would need to handle.

## 9. Duplicate Handling

### Concept Explanation
- `duplicated()` returns a boolean Series flagging rows that are duplicates of an earlier row (by default, based on **all** columns).
- `drop_duplicates()` removes those duplicate rows, optionally based on a **subset** of columns and keeping the `first`, `last`, or `False` (drop all copies).

### Business + AI/ML Example
Recall our dataset had customer **108 (Divya)** appear twice due to a data export glitch. The business needs a clean, one-row-per-customer table for accurate reporting (otherwise Divya's spend gets double-counted). Duplicate rows are equally dangerous for ML — they can cause **data leakage** between training and test sets if not removed first.

In [9]:
duplicate_flags = df.duplicated(subset=["customer_id"])
df[duplicate_flags]

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk,customer_tier,churn_risk_score
10,108,Divya,West,Gold,35,87000,3900,Low,Platinum Candidate,1


**Output Explanation:** `duplicated(subset=["customer_id"])` compared rows using only the `customer_id` column and flagged the **second occurrence** of customer 108 (Divya) as `True` (a duplicate), since `duplicated()` keeps the first occurrence as `False` by default.

In [10]:
df = df.drop_duplicates(subset=["customer_id"], keep="first")
df.reset_index(drop=True, inplace=True)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk,customer_tier,churn_risk_score
0,101,Aarav,South,Gold,42,125000,3200,Low,Platinum Candidate,1
1,102,Priya,North-East Zone,Silver,15,32000,1500,Medium,Silver,2
2,103,Rahul,South,Gold,30,98000,4200,Low,Gold,1
3,105,Vikram,North-East Zone,Silver,22,45000,2100,Medium,Rising Silver/Bronze,2
4,106,Ananya,South,Gold,60,210000,5000,Low,Platinum Candidate,1
5,107,Karthik,North-East Zone,Bronze,8,12000,1100,High,Bronze,3
6,108,Divya,West,Gold,35,87000,3900,Low,Platinum Candidate,1
7,109,Manoj,North-East Zone,Silver,18,39000,1800,Medium,Silver,2
8,110,Lakshmi,South,Bronze,3,4500,700,High,Bronze,3


**Output Explanation:** The duplicate row for Divya (customer 108) is gone — we now have exactly one row per unique `customer_id`. We also called `reset_index(drop=True)` to renumber the index cleanly (0 to n-1) after all the row drops earlier in the notebook, which keeps the DataFrame tidy for any downstream `loc`-based access or export.